# PharmaLens AI — Project 16
# Pharma Events & Activities Intelligence & Opportunity Engine

**Purpose:** Build a production-ready data intelligence notebook for pharmaceutical company events and activities.

The notebook creates a normalized event intelligence dataset suitable for later integration with PharmaLens AI backend, database, dashboards, AI Copilot, company subscriptions, alerts, and an event marketplace.

### Core capabilities
- Event/activity ingestion from CSV/XLSX/JSON/API-ready sources
- Standardization of event type, date, location, company, organizer, therapeutic area, audience, and budget
- Budget provenance: Actual / Reported / Estimated / Inferred / Unknown
- Source URL, verification date, confidence score, and evidence fields
- Company ↔ event relationships
- Therapeutic-area and geographic intelligence
- Company activity profiling
- Event opportunity scoring
- Future-event and alert-ready fields
- Parquet outputs designed for Supabase/PostgreSQL ingestion

> **Important:** Never present estimated or inferred budgets as actual company spending. Preserve provenance and confidence for every externally sourced field.


In [ ]:
# 1. Configuration & imports
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
from datetime import datetime

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed"
INPUT_DIR = DATA_DIR / "external" / "events"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Canonical Event Data Model

The master event table uses a stable schema so it can later map cleanly into PostgreSQL/Supabase.

### Main fields
- `event_id`
- `activity_name`
- `activity_type`
- `start_date`, `end_date`
- `country`, `city`, `venue`
- `organizer`
- `company_name`
- `participation_type`
- `therapeutic_area`
- `sub_therapeutic_area`
- `target_audience`
- `description`
- `budget_amount`
- `budget_currency`
- `budget_status`
- `budget_source`
- `budget_source_url`
- `budget_confidence`
- `source_name`, `source_url`
- `source_last_verified`
- `event_status`
- `is_future_event`
- `data_quality_score`
- `event_opportunity_score`


In [ ]:
# 3. Canonical schema and controlled vocabularies

CANONICAL_COLUMNS = [
    "event_id", "activity_name", "activity_type",
    "start_date", "end_date",
    "country", "city", "venue",
    "organizer", "company_name", "participation_type",
    "therapeutic_area", "sub_therapeutic_area",
    "target_audience", "description",
    "budget_amount", "budget_currency", "budget_status",
    "budget_source", "budget_source_url", "budget_confidence",
    "source_name", "source_url", "source_last_verified",
    "event_status", "is_future_event",
    "data_quality_score", "event_opportunity_score"
]

ACTIVITY_TYPES = [
    "Congress", "Conference", "Exhibition", "Symposium", "Workshop",
    "CME/CPD", "Medical Education", "Advisory Board", "Round Table",
    "KOL Meeting", "Product Launch", "Patient Awareness",
    "HCP Engagement", "Sponsorship", "Hospital Activity",
    "Pharmacy Activity", "Scientific Meeting", "Other", "Unknown"
]

PARTICIPATION_TYPES = [
    "Organizer", "Sponsor", "Exhibitor", "Speaker", "Partner",
    "Attendee", "Supporting Company", "Unknown"
]

BUDGET_STATUSES = [
    "Actual", "Reported", "Estimated", "Inferred", "Unknown"
]

EVENT_STATUSES = [
    "Planned", "Confirmed", "Completed", "Cancelled", "Postponed", "Unknown"
]

COLUMN_ALIASES = {
    "event": "activity_name",
    "event_name": "activity_name",
    "activity": "activity_name",
    "name": "activity_name",
    "type": "activity_type",
    "date": "start_date",
    "event_date": "start_date",
    "start": "start_date",
    "end": "end_date",
    "location": "city",
    "company": "company_name",
    "pharma_company": "company_name",
    "manufacturer": "company_name",
    "organizer_name": "organizer",
    "therapy_area": "therapeutic_area",
    "ta": "therapeutic_area",
    "budget": "budget_amount",
    "budget_value": "budget_amount",
    "currency": "budget_currency",
    "source": "source_name",
    "url": "source_url",
}


In [ ]:
# 4. Utility functions

def normalize_col(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col).strip("_")
    return col

def standardize_columns(df):
    out = df.copy()
    out.columns = [normalize_col(c) for c in out.columns]
    rename = {c: COLUMN_ALIASES[c] for c in out.columns if c in COLUMN_ALIASES}
    out = out.rename(columns=rename)
    return out

def ensure_columns(df):
    out = df.copy()
    for col in CANONICAL_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    return out[CANONICAL_COLUMNS]

def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = re.sub(r"\s+", " ", str(x)).strip()
    return x if x else np.nan

def normalize_text_columns(df):
    out = df.copy()
    for col in out.select_dtypes(include=["object"]).columns:
        out[col] = out[col].map(clean_text)
    return out

def parse_dates(df):
    out = df.copy()
    for col in ["start_date", "end_date", "source_last_verified"]:
        out[col] = pd.to_datetime(out[col], errors="coerce")
    return out

def normalize_budget(df):
    out = df.copy()
    out["budget_amount"] = pd.to_numeric(
        out["budget_amount"].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
    out["budget_currency"] = out["budget_currency"].fillna("EGP")
    out["budget_status"] = out["budget_status"].fillna("Unknown")
    return out

def make_event_id(row):
    if pd.notna(row.get("event_id")) and str(row["event_id"]).strip():
        return str(row["event_id"])
    base = "|".join([
        str(row.get("activity_name", "")),
        str(row.get("start_date", "")),
        str(row.get("city", "")),
        str(row.get("organizer", "")),
    ])
    import hashlib
    return "EVT-" + hashlib.sha1(base.encode("utf-8")).hexdigest()[:12].upper()


In [ ]:
# 5. Load source files
# Put CSV/XLSX/JSON files inside:
# data/external/events/

def load_event_files(input_dir=INPUT_DIR):
    frames = []
    files = list(input_dir.glob("*.csv")) + list(input_dir.glob("*.xlsx")) + list(input_dir.glob("*.xls")) + list(input_dir.glob("*.json"))

    for path in files:
        try:
            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path)
            elif path.suffix.lower() in [".xlsx", ".xls"]:
                df = pd.read_excel(path)
            else:
                with open(path, "r", encoding="utf-8") as f:
                    obj = json.load(f)
                df = pd.DataFrame(obj if isinstance(obj, list) else obj.get("data", obj))

            df["__input_file"] = path.name
            frames.append(df)
            print(f"Loaded: {path.name} | rows={len(df):,}")
        except Exception as e:
            print(f"FAILED: {path.name} -> {e}")

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)

raw_events = load_event_files()
print("Raw rows:", len(raw_events))


In [ ]:
# 6. Standardize and clean

if raw_events.empty:
    events = pd.DataFrame(columns=CANONICAL_COLUMNS)
else:
    events = standardize_columns(raw_events)
    events = normalize_text_columns(events)
    events = ensure_columns(events)
    events = parse_dates(events)
    events = normalize_budget(events)

    # Controlled vocabulary cleanup
    events["activity_type"] = events["activity_type"].fillna("Unknown")
    events["participation_type"] = events["participation_type"].fillna("Unknown")
    events["event_status"] = events["event_status"].fillna("Unknown")
    events["therapeutic_area"] = events["therapeutic_area"].fillna("Unknown")
    events["country"] = events["country"].fillna("Egypt")

    events["event_id"] = events.apply(make_event_id, axis=1)
    events["is_future_event"] = events["start_date"] > pd.Timestamp.now().normalize()

events.head()


## 7. Data quality & provenance

Every event should be traceable to a source.

Recommended source hierarchy:
1. Official organizer/congress website
2. Pharmaceutical company official announcement
3. Hospital/university/medical society
4. Recognized professional association
5. Reputable event directory
6. Secondary/public reporting

Budget values require stricter provenance. If a budget cannot be verified, keep it as `Estimated`, `Inferred`, or `Unknown`.


In [ ]:
# 8. Data quality scoring

def calculate_quality_score(row):
    score = 0

    required = ["activity_name", "activity_type", "start_date", "city", "organizer", "therapeutic_area"]
    score += sum(pd.notna(row.get(c)) and str(row.get(c)).strip() not in ["", "nan"] for c in required) * 10

    if pd.notna(row.get("source_url")):
        score += 15
    if pd.notna(row.get("source_name")):
        score += 10
    if pd.notna(row.get("source_last_verified")):
        score += 10
    if row.get("budget_status") in ["Actual", "Reported"]:
        score += 15
    elif row.get("budget_status") in ["Estimated", "Inferred"]:
        score += 5

    return min(score, 100)

if not events.empty:
    events["data_quality_score"] = events.apply(calculate_quality_score, axis=1)
else:
    events["data_quality_score"] = pd.Series(dtype=float)

events[["event_id", "activity_name", "data_quality_score"]].head(10)


In [ ]:
# 9. Event Opportunity Score
# A transparent baseline score; weights can later be learned/calibrated from customer outcomes.

def score_event(row):
    score = 0.0

    # Therapeutic relevance
    ta = str(row.get("therapeutic_area", "")).lower()
    if ta and ta not in ["unknown", "nan"]:
        score += 20

    # Audience relevance
    audience = str(row.get("target_audience", "")).lower()
    audience_terms = ["physician", "doctor", "hcp", "psychiatrist", "pharmacist", "oncologist", "cardiologist"]
    if any(term in audience for term in audience_terms):
        score += 20

    # Company participation
    if pd.notna(row.get("company_name")):
        score += 15

    # Location completeness
    if pd.notna(row.get("city")):
        score += 10

    # Source quality
    score += min(float(row.get("data_quality_score", 0)) * 0.20, 20)

    # Future event bonus
    if bool(row.get("is_future_event", False)):
        score += 15

    return round(min(score, 100), 2)

if not events.empty:
    events["event_opportunity_score"] = events.apply(score_event, axis=1)
else:
    events["event_opportunity_score"] = pd.Series(dtype=float)

events[["activity_name", "therapeutic_area", "city", "event_opportunity_score"]].sort_values(
    "event_opportunity_score", ascending=False
).head(20)


In [ ]:
# 10. Build company-event relationship table

company_events = events[
    ["event_id", "company_name", "participation_type", "budget_amount",
     "budget_currency", "budget_status", "budget_source", "budget_source_url",
     "budget_confidence"]
].copy()

company_events = company_events.dropna(subset=["company_name"]).drop_duplicates()

# Company activity intelligence
company_activity = (
    company_events.groupby("company_name", dropna=True)
    .agg(
        total_events=("event_id", "nunique"),
        total_budget_reported=("budget_amount", "sum"),
        average_event_budget=("budget_amount", "mean"),
        therapeutic_areas=("event_id", "count"),
    )
    .reset_index()
    .rename(columns={"therapeutic_areas": "company_event_records"})
)

company_activity.head(20)


In [ ]:
# 11. Therapeutic-area intelligence

ta_intelligence = (
    events.groupby("therapeutic_area", dropna=False)
    .agg(
        total_events=("event_id", "nunique"),
        companies_active=("company_name", "nunique"),
        cities=("city", "nunique"),
        future_events=("is_future_event", "sum"),
        avg_opportunity_score=("event_opportunity_score", "mean"),
        total_reported_or_estimated_budget=("budget_amount", "sum"),
    )
    .reset_index()
    .sort_values(["future_events", "total_events"], ascending=False)
)

ta_intelligence.head(30)


In [ ]:
# 12. Geographic intelligence

geo_intelligence = (
    events.groupby(["country", "city"], dropna=False)
    .agg(
        total_events=("event_id", "nunique"),
        companies_active=("company_name", "nunique"),
        therapeutic_areas=("therapeutic_area", "nunique"),
        future_events=("is_future_event", "sum"),
        avg_opportunity_score=("event_opportunity_score", "mean"),
    )
    .reset_index()
    .sort_values("total_events", ascending=False)
)

geo_intelligence.head(30)


In [ ]:
# 13. Future event calendar / alert-ready dataset

future_events = events[
    events["is_future_event"] == True
].copy()

future_events = future_events.sort_values(["start_date", "event_opportunity_score"], ascending=[True, False])

future_events[[
    "event_id", "activity_name", "activity_type", "start_date", "end_date",
    "country", "city", "organizer", "company_name", "therapeutic_area",
    "target_audience", "event_opportunity_score", "source_url"
]].head(50)


## 14. Subscription / alert rules

This dataset is intentionally designed to support future company subscriptions.

Example subscription:
```text
Company: Example Pharma
Country: Egypt
Therapeutic Areas: Psychiatry, Neurology
Event Types: Congress, CME/CPD, Workshop
Cities: Cairo, Giza, Alexandria
Notify: Email + In-app
Minimum opportunity score: 70
```

The notebook does not send notifications. It creates the normalized event and matching data that the backend can use for alerts.


In [ ]:
# 15. Example rule matcher for future backend integration

def match_event(event, rule):
    if rule.get("country") and str(event.get("country")).lower() != str(rule["country"]).lower():
        return False

    tas = rule.get("therapeutic_areas", [])
    if tas:
        event_ta = str(event.get("therapeutic_area", "")).lower()
        if not any(str(x).lower() in event_ta for x in tas):
            return False

    types = rule.get("activity_types", [])
    if types and event.get("activity_type") not in types:
        return False

    cities = rule.get("cities", [])
    if cities and str(event.get("city", "")).lower() not in [str(x).lower() for x in cities]:
        return False

    minimum_score = rule.get("minimum_opportunity_score")
    if minimum_score is not None and float(event.get("event_opportunity_score", 0)) < minimum_score:
        return False

    return True

example_rule = {
    "country": "Egypt",
    "therapeutic_areas": ["Psychiatry"],
    "activity_types": ["Congress", "CME/CPD", "Workshop"],
    "cities": [],
    "minimum_opportunity_score": 70,
}

if not future_events.empty:
    matches = future_events[future_events.apply(lambda r: match_event(r, example_rule), axis=1)]
else:
    matches = future_events.copy()

matches.head(20)


In [ ]:
# 16. Export production-ready datasets

exports = {
    "fact_pharma_events": events,
    "dim_event": events[[
        "event_id", "activity_name", "activity_type", "start_date", "end_date",
        "country", "city", "venue", "organizer", "therapeutic_area",
        "sub_therapeutic_area", "target_audience", "description",
        "event_status", "is_future_event", "data_quality_score",
        "event_opportunity_score", "source_name", "source_url",
        "source_last_verified"
    ]].copy(),
    "dim_event_company": company_events,
    "fact_event_budget": company_events[[
        "event_id", "company_name", "budget_amount", "budget_currency",
        "budget_status", "budget_source", "budget_source_url",
        "budget_confidence"
    ]].copy(),
    "fact_company_activity": company_activity,
    "fact_therapeutic_area_activity": ta_intelligence,
    "fact_geographic_activity": geo_intelligence,
    "fact_future_events": future_events
}

for name, df in exports.items():
    path = OUTPUT_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved {path} | rows={len(df):,}")

print("\nExport directory:", OUTPUT_DIR)


## 17. Recommended database mapping

For Supabase/PostgreSQL, map the outputs approximately as:

```text
events
event_companies
event_budgets
event_sources
companies
therapeutic_areas
locations
event_subscriptions
event_alerts
```

### Future `event_subscriptions` fields
- `subscription_id`
- `company_id`
- `country`
- `cities`
- `therapeutic_areas`
- `activity_types`
- `competitors`
- `minimum_opportunity_score`
- `notification_channels`
- `is_active`
- `created_at`

### Future AI Copilot actions
- Search events
- Compare companies' event activity
- Find competitor activity
- Find future events
- Recommend sponsorship opportunities
- Explain event opportunity score
- Create an event shortlist
- Create subscription/alert criteria


## 18. Data-source / API adapter pattern

The notebook is intentionally source-agnostic. A future connector can populate the same canonical schema from:
- official event/congress websites
- medical societies
- hospitals/universities
- pharmaceutical company announcements
- licensed event databases/APIs
- customer-provided event files
- approved third-party data providers

For production, each ingestion adapter should preserve:
`source_name`, `source_url`, `source_last_verified`, `retrieval_timestamp`, and field-level confidence where possible.

Do not scrape or redistribute data in violation of a website's terms, license, robots policy, or applicable law.


In [ ]:
# 19. Final validation report

validation = {
    "rows": len(events),
    "unique_events": events["event_id"].nunique() if not events.empty else 0,
    "future_events": int(events["is_future_event"].sum()) if not events.empty else 0,
    "companies": events["company_name"].nunique(dropna=True) if not events.empty else 0,
    "therapeutic_areas": events["therapeutic_area"].nunique(dropna=True) if not events.empty else 0,
    "cities": events["city"].nunique(dropna=True) if not events.empty else 0,
    "events_with_source_url": int(events["source_url"].notna().sum()) if not events.empty else 0,
    "events_with_budget": int(events["budget_amount"].notna().sum()) if not events.empty else 0,
    "avg_data_quality_score": float(events["data_quality_score"].mean()) if not events.empty else 0,
    "avg_opportunity_score": float(events["event_opportunity_score"].mean()) if not events.empty else 0,
}

pd.Series(validation)
